In [1]:
import pandas as pd

df = pd.read_json('data/ReposVul.jsonl', lines=True)
display(df.shape)
df.columns

(6897, 30)

Index(['index', 'cve_id', 'cwe_id', 'cve_language', 'cve_description', 'cvss',
       'publish_date', 'AV', 'AC', 'PR', 'UI', 'S', 'C', 'I', 'A', 'commit_id',
       'commit_message', 'commit_date', 'project', 'url', 'html_url',
       'windows_before', 'windows_after', 'parents', 'details', 'outdated',
       'cwe_descripiton', 'cwe_consequence', 'cwe_method', 'cwe_solution'],
      dtype='object')

In [26]:
new_df = []
for index, row in df.iterrows():
    # skip outdated records (이 커밋 이후에도 취약점 수정이 이루어진 경우 스킵)
    outdated = row['outdated']
    if outdated == 1: continue
    
    cve_id = row['cve_id']
    cwe_id = row['cwe_id']
    cvss = row['cvss']
    language = row['cve_language'].lower()
    project = row['project']
    commit_id = row['commit_id']
    parents = row['parents']
    commit_id_before = parents[-1]['commit_id_before']
    details = row['details']
    
    # filter if multiple files are changed in a single commit
    # if len(details) > 1: continue
    
    data = []
    for detail in details:
        # CVE 언어와 파일 언어가 다른 경우 스킵
        if "file_language" not in detail:
            file_ext = detail["file_language"].lower()
            if language == "python" :
                if file_ext != "py":
                    continue
            elif language == "c++":
                if file_ext != "cpp":
                    continue
            elif language == "c":
                if file_ext != "c":
                    continue
            elif language == "java":
                if file_ext != "java":
                    continue
            continue
        
        # filter only single function changes
        function_before = detail.get("function_before", [])
        vul_functions = []
        for func in function_before:
            if func["target"] == 1:
                vul_functions.append(func)
        if len(vul_functions) != 1: continue
        
        fix_functions = []
        function_after = detail.get("function_after", [])
        for func in function_after:
            if func["target"] == 1:
                continue
            has_same_func = False
            func_after = func["function"]
            for func_b in function_before:
                if func_b["function"] == func_after:
                    has_same_func = True
                    break
            if not has_same_func:
                fix_functions.append(func)
        if len(fix_functions) != 1: continue
        
        # Add single file with single vulnerable function changes to new_df
        vul_func = vul_functions[0]
        data.append({
            'cve_id': cve_id,
            'cwe_id': tuple(cwe_id),
            'language': language,
            'project': project,
            'commit_id': commit_id_before,
            'file_name': detail['file_name'],
            'line': vul_func['line'],
            'function': vul_func["function"],
            'file': detail["code_before"],
            'repository': None,
            'vulnerable': True
        })
        
        non_vul_func = fix_functions[0]
        data.append({
            'cve_id': cve_id,
            'cwe_id': tuple(cwe_id),
            'language': language,
            'project': project,
            'commit_id': commit_id,
            'file_name': detail['file_name'],
            'line': None,
            'function': non_vul_func["function"],
            'file': detail["code"],
            'repository': None,
            'vulnerable': False
        })
    
    if len(data) == 2:
        new_df.extend(data)
new_df = pd.DataFrame(new_df)

# project별로 정렬하고 같은 project, commit_id 묶어서 처리 속도 향상
new_df = new_df.sort_values(by=['project', 'commit_id']).reset_index(drop=True)
new_df = new_df.drop_duplicates().reset_index(drop=True)
display(new_df.shape)

(572, 11)

In [27]:
old_df = pd.read_json('data/FuncFileRepo.jsonl', lines=True)
display(old_df.shape)

columns = new_df.columns
for idx, row in new_df.iterrows():
    for _, old_row in old_df.iterrows():
        is_same = True
        for col in columns:
            if col == 'repository':
                continue
            if col == 'cwe_id':
                if list(row[col]) != list(old_row[col]):
                    is_same = False
                    break
                continue
            if row[col] != old_row[col]:
                is_same = False
                break
        if is_same:
            new_df.at[idx, 'repository'] = old_row['repository']
            break
new_df.shape

(322, 11)

(572, 11)

In [33]:
# new_df의 repository 컬럼에 값이 '{"callee": [], "caller": []}' 인 경우 제거
new_df = new_df[new_df['repository'] != "{'callee': [], 'caller': []}"]
new_df = new_df.reset_index(drop=True)
display(new_df.shape)
# new_df의 repository 컬럼의 값이 None인 경우 개수
new_df['repository'].isnull().sum()

new_df.to_json('data/FuncFileRepo2.jsonl', orient='records', lines=True)

(515, 11)

In [ ]:
import os
import pandas as pd
import subprocess
from pathlib import Path
from src.utils import FuncNameParser
from tqdm.notebook import tqdm

# ====== 설정 ======
BASE_URL = "https://github.com"
BASE_DIR = Path("codeql").resolve()
DEFAULT_ENV = {
    "GIT_LFS_SKIP_SMUDGE": "1",   # 체크아웃/리셋 시 LFS 객체 다운로드 금지
    "GIT_TERMINAL_PROMPT": "0",   # 인증 프롬프트 방지
}
# ==================

def run(cmd, cwd=None, ignore_error=False, env=None, timeout=None):
    merged_env = os.environ.copy()
    merged_env.update(DEFAULT_ENV)
    if env:
        merged_env.update(env)
    r = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
        env=merged_env,
        timeout=timeout,
    )
    if r.returncode != 0 and not ignore_error:
        details = "[cmd failed] {}\nSTDOUT:\n{}\nSTDERR:\n{}".format(
            " ".join(cmd),
            r.stdout,
            r.stderr,
        )
        raise RuntimeError(details)
    return r.stdout.strip()

def safe_name(owner_repo: str) -> str:
    return owner_repo.replace("/", "__")

def disable_lfs(repo_dir: Path):
    """
    (최대한 안전하게) 현재 저장소에서 LFS 필터 자체를 비활성화합니다.
    git-lfs 미설치/오류 상황에서도 체크아웃이 계속되도록 합니다.
    """
    # filter.lfs.process를 비워서 외부 필터 호출 자체를 막음
    run(["git", "-C", str(repo_dir), "config", "--local", "filter.lfs.process", ""], ignore_error=True)
    # 필수 아님으로 지정
    run(["git", "-C", str(repo_dir), "config", "--local", "filter.lfs.required", "false"], ignore_error=True)
    # smudge 단계 대체(다운로드 대신 통과); 일부 git 버전에서 필요
    run(["git", "-C", str(repo_dir), "config", "--local", "filter.lfs.smudge", "cat"], ignore_error=True)
    # LFS fetch도 전부 제외
    run(["git", "-C", str(repo_dir), "config", "--local", "lfs.fetchexclude", "*"], ignore_error=True)

def ensure_repo(owner_repo: str, language: str) -> Path:
    """
    프로젝트 작업 디렉토리가 없으면 최초 1회만 클론합니다.
    이후에는 같은 디렉토리를 재사용합니다.
    """
    repo_dir = BASE_DIR / language / safe_name(owner_repo)
    if not repo_dir.exists():
        repo_url = f"{BASE_URL.rstrip('/')}/{owner_repo}.git"
        # print(f"[clone] {repo_url} -> {repo_dir}")
        # 부분 클론으로 최초 트래픽/공간 최소화 (필요한 blob은 체크아웃 시점에 on-demand로 받음)
        run(["git", "clone", "--filter=blob:none", "--no-tags", repo_url, str(repo_dir)])
        # 기본 브랜치를 알 수 없으니 일단 패치 대상 커밋별로 가져와서 체크아웃할 예정
    # else:
    #     print(f"[reuse] {repo_dir}")
    disable_lfs(repo_dir)
    return repo_dir

def checkout_commit(repo_dir: Path, commit_id: str):
    """
    작업 디렉토리를 해당 커밋 상태로 깔끔하게 맞춥니다.
    (detached HEAD + 하드 리셋 + 쓰레기 파일 제거)
    """
    disable_lfs(repo_dir)
    ensure_commit_fetched(repo_dir, commit_id)

    # 워킹트리 변경사항/빌드 산출물 제거
    run(["git", "-C", str(repo_dir), "reset", "--hard"])
    run(["git", "-C", str(repo_dir), "clean", "-fdx"])

    # 커밋으로 이동(detached HEAD)
    run(["git", "-C", str(repo_dir), "checkout", "--force", "--detach", commit_id])
    run(["git", "-C", str(repo_dir), "reset", "--hard", commit_id])
    run(["git", "-C", str(repo_dir), "clean", "-fdx"])

    # 서브모듈 쓰는 레포 대비
    run(["git", "-C", str(repo_dir), "submodule", "update", "--init", "--recursive"], ignore_error=True)

def ensure_commit_fetched(repo_dir: Path, commit_id: str):
    """
    해당 커밋이 로컬에 없으면 최소한의 깊이로 정확히 그 커밋만 fetch 합니다.
    """
    try:
        run(["git", "-C", str(repo_dir), "rev-parse", "--verify", commit_id])
    except RuntimeError:
        # 지정 커밋만 얕게 가져와서 히스토리 팽창을 막습니다.
        run(["git", "-C", str(repo_dir), "fetch", "origin", commit_id, "--depth=1"])
        # 검증 재시도
        run(["git", "-C", str(repo_dir), "rev-parse", "--verify", commit_id])


# ================= 사용 예시 =================
save_path = "data/FuncFileRepo.jsonl"
before_projects = None
before_commit = None

if os.path.exists(save_path):
    new_df = pd.read_json(save_path, lines=True)

for index, row in tqdm(new_df.iterrows(), total=len(new_df)):
    project = row["project"]
    language = row["language"]
    if language == "c++": language = "cpp"
    commit_id = row["commit_id"]
    file_name = row["file_name"]
    function = row["function"]
    func_name = FuncNameParser.run(function, language)
    
    logs = []
    
    if func_name is None or \
        func_name.strip() == "" or \
        "\n" in func_name or \
        " " in func_name:
        logs.append(f"[skip] 함수 이름을 추출할 수 없습니다: {project}@{commit_id[:12]} - {file_name}")
        continue
    
    if row["repository"] is not None and row["repository"] == "{'callee': [], 'caller': []}":
        continue  # 이미 처리된 항목은 건너뜀
    try:
        repo = f"codeql/{language}/{project.replace('/', '__')}"
        script_dir = os.path.abspath(os.path.dirname(repo))
        
        # if project == "bminor/binutils-gdb" and commit_id == "d12f8998d2d086f0a6606589e5aedb7147e6f2f1":
        #     print("debug")
        #     pass
        # else:
        #     continue
        
        if before_projects != project or before_commit != commit_id:
            repo_dir = ensure_repo(project, language)
            checkout_commit(repo_dir, commit_id)
            logs.append(f"[ready] {project}@{commit_id[:12]} -> {repo_dir}")
            
            build_shell = open(os.path.join("codeql", language, "build.sh"), "r").read()
            build_script = build_shell.format(script_dir=script_dir, repo=repo)
            subprocess.run(
                ["bash", "-lc", build_script],
                text=True, capture_output=True, check=True
                )
            logs.append("[build] Database created.")
        
        calls_ql = open(os.path.join("codeql", language, "template.ql"), "r").read()
        calls_ql = calls_ql.replace('string targetFileName()      { result = "" }',
                                    "string targetFileName()      { result = \"" + file_name + "\" }")
        calls_ql = calls_ql.replace('string targetFunctionName()  { result = "" }',
                                    "string targetFunctionName()  { result = \"" + func_name + "\" }")
        with open(os.path.join(script_dir, "calls.ql"), "w") as f:
            f.write(calls_ql)
        logs.append("[prep] Query prepared.")
        
        run_shell = open(os.path.join("codeql", language, "run.sh"), "r").read()
        run_script = run_shell.format(script_dir=script_dir)
        calls = subprocess.run(
            ["bash", "-lc", run_script], 
            text=True, capture_output=True, check=True,
            cwd=script_dir)
        logs.append("[run] Query executed.")
        new_df.at[index, "repository"] = calls.stdout.strip()

    except Exception as e:
        print("\n".join(logs))
        print(file_name)
        print(func_name)
        print(function)
        print(f"[error] {project}@{commit_id[:12]}: {e}")
        # stderr도 출력하면 디버깅에 도움됨
        if hasattr(e, 'stderr'):
            print(f"[stderr] {e.stderr}")
        break
        # pass
    
    before_projects, before_commit = project, commit_id
    new_df.to_json(save_path, orient='records', lines=True)
    

In [ ]:
new_df.to_json('data/FuncFileRepo.jsonl', orient='records', lines=True)

In [46]:
import pandas as pd

new_df = pd.read_json("data/FuncFileRepo2.jsonl", lines=True)

# print(len(new_df))
# print(len(new_df['cve_id'].unique()))
# # 튜플로 된 cwe_id를 개별 요소를 추출하여 고유한 cwe_id 개수 계산
# unique_cwe_ids = set()
# for cwe_tuple in new_df['cwe_id']:
#     unique_cwe_ids.update(cwe_tuple)
# print(len(unique_cwe_ids))
# print(len(new_df['project'].unique()))
# new_df['language'].value_counts()

# new_df['repository'] 중 None 값 확인
# new_df['repository'].isnull().sum()
# new_df[new_df['repository'].isnull()]

# new_df['repository'] 중 None을 가진 행들 출력
# print(new_df[new_df['repository'].isnull()].shape)
# new_df[new_df['repository'] == "{'callee': [], 'caller': []}"]

# new_df['commit_id] == 'd12f8998d2d086f0a6606589e5aedb7147e6f2f1' 인 행 출력
# new_df[new_df['commit_id'] == 'd12f8998d2d086f0a6606589e5aedb7147e6f2f1']

new_df.iloc[20]

# unique = {"cve_id": set(), "cwe_id": set(), "project": set(), "language": {}}
# for index, row in new_df.iterrows():
#     cve_id = row["cve_id"]
#     cwe_ids = row["cwe_id"]
#     project = row["project"]
#     language = row["language"]
#     repository = row["repository"]
#     vernerable = row["vulnerable"]
    
#     if repository is None or repository == "{'callee': [], 'caller': []}":
#         display(row)
#         continue
    
#     unique["cve_id"].add(cve_id)
#     for cwe_id in cwe_ids:
#         unique["cwe_id"].add(cwe_id)
#     unique["project"].add(project)
#     if language not in unique["language"]:
#         unique["language"][language] = 0
#     unique["language"][language] += 1

# for key, value in unique.items():
#     if key == "language":
#         print(f"{key}:")
#         for lang, count in value.items():
#             print(f"  {lang}: {count}")
#     else:
#         print(f"{key}: {len(value)}")



cve_id                                           CVE-2017-14639
cwe_id                                       [CWE-119, CWE-843]
language                                                    c++
project                                axiomatic-systems/bento4
commit_id              0191f86015ca545630d561887b4d6a5785075ed5
file_name                    Source/C++/Core/Ap4SampleEntry.cpp
line          @@  -772,13 +772,13  @@ AP4_VisualSampleEntry:...
function      AP4_Result\nAP4_VisualSampleEntry::ReadFields(...
file          /*********************************************...
repository    {'callee': [{'file': 'Source/C++/Core/Ap4DataB...
vulnerable                                                 True
Name: 20, dtype: object

In [ ]:
# new_df 165번쨰 삭제
new_df = new_df.drop(index=165).reset_index(drop=True)
new_df2 = new_df[(new_df['repository'].notnull()) & (new_df['repository'] != "{'callee': [], 'caller': []}")]
new_df2.to_json('data/FuncFileRepo_Cleaned.jsonl', orient='records', lines=True)

In [ ]:
import pandas as pd

df = pd.read_json('data/FuncFileRepo_Cleaned.jsonl', lines=True)
calls = df.iloc[1]['repository']

import ast
calls_dict = ast.literal_eval(calls)
print(calls_dict['callee'])

In [ ]:
from tree_sitter import Query, QueryCursor
from tree_sitter_language_pack import get_language, get_parser

def print_tree(node, code, indent=0):
    """AST 트리 출력"""
    text = code[node.start_byte:node.end_byte]
    if len(text) > 50:
        text = text[:50] + "..."
    print("  " * indent + f"{node.type}: {repr(text)}")
    for child in node.children:
        print_tree(child, code, indent + 1)

code = """class FileTaskHandler(logging.Handler):
    pass"""

parser = get_parser("python")
tree = parser.parse(code.encode("utf-8"))

print("=== AST 구조 ===")
print_tree(tree.root_node, code)

In [ ]:
from src.utils import FuncNameParser

code = '''def search(request):
	context_dict = {}
	if 'q' in request.GET and request.GET['q'] != '':
		q = request.GET['q']
		cursor = connection.cursor()
		cursor.execute("SELECT id,title,artist,cover FROM recordstoreapp_record WHERE title like '%" + q + "%' or artist like '%" + q + "%' or label like '%" + q + "%' or cat_no like '%" + q + "%';")
		rec_list=cursor.fetchall()
		
		total=len(rec_list)
		pg=int(request.GET['page']) if 'page' in request.GET else 1
		ub=min(pg*12, total)

		context_dict['rec_list'] = rec_list[(pg-1)*12:ub]
		maxrange = int(total/12)
		if total%12 > 0: 
			maxrange = maxrange + 1
		if maxrange == 1: 
			maxrange = 0
		context_dict['range'] = range(1,maxrange+1)
		print total
		context_dict['q'] = q

	return render(request, 'search.html', context_dict)'''
        
func_name = FuncNameParser.run(code, "python")
print(func_name)

In [ ]:
file_prompt_file:str="src/prompts/detection/file.md"

from src.prompts import PromptManager

pm = PromptManager()
user = pm.render(
    file=file_prompt_file,
    language="python",
    function="def search(request): pass",
    file_code="# import pandas as pd\n"
)
print(user)